# Week 9 — Cross-City Evaluation (STARTER)
### The Computer Vision Internship | VisionAI Lagos

## Cross-City Transfer Evaluation

Amara: 'The overall headline F1 may be hiding something. Today we find out.'

> ⏸️ **Pause and Predict**
> Which city will have the lowest F1? Which class will be most affected? Write your prediction — this is your Week 2 hypothesis test.

In [ ]:
# Load best model (use EfficientNet from Week 8)
def build_efficientnet_b0(nc=4,pretrained=False):
    m=models.efficientnet_b0(weights=None)
    m.classifier=nn.Sequential(nn.Dropout(0.3),nn.Linear(1280,nc))
    return m

model=build_efficientnet_b0().to(DEVICE)
model.load_state_dict(torch.load("models/efficientnet_best.pth",map_location=DEVICE))
model.eval(); print("Model loaded ✅")

In [ ]:
# Per-city evaluation — the headline test
print("=== PER-CITY PERFORMANCE ===\n")
results={}
ev=transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
test_df=df[df["split"]=="test"]
for city in sorted(test_df["city"].unique()):
    sub=test_df[test_df["city"]==city]
    loader=DataLoader(SatelliteDataset(sub,"datasets/images",ev),64,False,num_workers=0)
    preds,labels=[],[]
    with torch.no_grad():
        for imgs,lbls,_ in loader:
            preds.extend(model(imgs.to(DEVICE)).argmax(1).cpu().numpy()); labels.extend(lbls.numpy())
    f1=f1_score(labels,preds,average="weighted",zero_division=0)
    results[city]={"f1":f1,"preds":preds,"labels":labels,"n":len(sub)}
    print(f"  {city:<8}: F1={f1:.3f}  (n={len(sub):,})")

best=max(results,key=lambda c:results[c]["f1"]); worst=min(results,key=lambda c:results[c]["f1"])
gap=results[best]["f1"]-results[worst]["f1"]
print(f"\nEquity gap: {gap:.3f}  ({best}={results[best]['f1']:.3f}  {worst}={results[worst]['f1']:.3f})")
print(f"{'Gap > 0.10 — fairness audit required (Week 10)' if gap>0.10 else 'Gap < 0.10 — acceptable'}")

In [ ]:
# Cross-city heatmap (per-city × per-class)
cities=sorted(results.keys())
f1_grid=np.zeros((len(cities),len(CLASSES)))
for i,city in enumerate(cities):
    from sklearn.metrics import classification_report as cr
    rep=cr(results[city]["labels"],results[city]["preds"],target_names=CLASSES,
           output_dict=True,zero_division=0)
    for j,cls in enumerate(CLASSES): f1_grid[i,j]=rep.get(cls,{}).get("f1",0)

fig,ax=plt.subplots(figsize=(9,6))
sns.heatmap(pd.DataFrame(f1_grid,index=cities,columns=CLASSES),
            annot=True,fmt=".2f",cmap="RdYlGn",vmin=0.4,vmax=1.0,ax=ax,
            linewidths=0.5,linecolor="white")
ax.set_title("Per-City x Per-Class F1 — Cross-City Transfer Evaluation\n(below 0.70 = remediation required)",
             fontweight="bold",loc="left")
plt.tight_layout(); plt.savefig("outputs/cross_city_heatmap.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()

## Qualitative Error Inspection

> ⏸️ **Recall Prompt**
> Book 2 Week 9 measured English → Yoruba transfer gap (ΔF1). Book 3 measures Lagos → Kano transfer gap (ΔF1). In what sense are these the same problem? In what sense are they different?

In [ ]:
# Look at wrong predictions — what visual property do Kano errors share?
from PIL import Image
pred_df=pd.DataFrame({"filename":[],"city":[],"true":[],"pred":[],"correct":[]})
rows_list=[]
ev2=transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
for city in cities:
    sub=test_df[test_df["city"]==city]
    for i,(true_l,pred_l) in enumerate(zip(results[city]["labels"],results[city]["preds"])):
        rows_list.append({"filename":sub.iloc[i]["filename"],"city":city,
                           "true":IDX2CLS[true_l],"pred":IDX2CLS[pred_l],"correct":true_l==pred_l})
pred_df=pd.DataFrame(rows_list)

fig,axes=plt.subplots(len(cities),6,figsize=(20,4*len(cities)))
for ri,city in enumerate(cities):
    correct=pred_df[(pred_df["city"]==city)&pred_df["correct"]].sample(min(3,sum((pred_df["city"]==city)&pred_df["correct"])),random_state=42)
    errors =pred_df[(pred_df["city"]==city)&~pred_df["correct"]].head(3)
    for ci,(idx,row) in enumerate(list(correct.iterrows())[:3]):
        img=Image.open(f"datasets/images/{row['filename']}")
        axes[ri][ci].imshow(img); axes[ri][ci].set_title(f"ok: {row['true'][:4]}",fontsize=8,color="green")
        axes[ri][ci].axis("off")
        if ci==0: axes[ri][ci].set_ylabel(f"{city}\ncorrect",rotation=0,labelpad=50,fontsize=8)
    for ci,(idx,row) in enumerate(list(errors.iterrows())[:3]):
        ax=axes[ri][ci+3]; img=Image.open(f"datasets/images/{row['filename']}")
        ax.imshow(img); ax.set_title(f"{row['true'][:4]}->{row['pred'][:4]}",fontsize=8,color="red")
        ax.axis("off")
        if ci==0: ax.set_ylabel(f"{city}\nerrors",rotation=0,labelpad=50,fontsize=8)
plt.suptitle("Qualitative Error Inspection — What do Kano errors look like?",fontweight="bold",y=1.01)
plt.tight_layout(); plt.savefig("outputs/qualitative_errors.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()
print("After viewing: write in a markdown cell what visual property Kano errors share.")

## Challenge Task

> **Quick arithmetic:** overall F1 = 0.90, Kano F1 = 0.75, four equal-size city splits.
> Solve: 0.75 + 3x = 4 x 0.90. What is the average F1 of the other three cities?
> Now: implement a domain adaptation baseline. Train ONLY on Lagos+Ibadan, evaluate on Kano-only test data. How much of Kano performance comes from Kano training examples vs shared visual features?

## Week 9 Complete

Next: `week10_fairness_STARTER.ipynb`

*The Computer Vision Internship · VisionAI Lagos*

## ✅ By completing Week 9, you can now:

- Evaluate the best model on each city separately
- Identify which city has the largest gap and explain why
- Build an error inspection log with image examples of each failure type
- Write a cross-city evaluation brief for a deployment team

📤 **GitHub:** Push week9_cross_city_STARTER.ipynb and cross_city_brief.md. Commit: "Week 9: Cross-city evaluation complete"


---

## 📚 Get the Full Book

This notebook is part of **The Computer Vision Internship** — Book 3 of the InternshipInABook™ Series.

👉 **[Get the book on Selar → [SELAR_LINK_PLACEHOLDER]]**

---
*InternshipInABook™ · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook/cv-internship-in-a-book)*
